# AI Lara — Qwen LoRA/QLoRA Starter
Notebook awal untuk menyiapkan lingkungan dan menguji Qwen2.5-1.5B-Instruct di Google Colab. Jalankan sel dari atas ke bawah. Pelatihan adapter memerlukan GPU. Notebook ini belum menghasilkan APK Android.

## 1. Periksa runtime dan GPU

In [ ]:
import sys, torch
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA tersedia:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Instal pustaka
Jalankan sekali. Jika Colab meminta restart runtime, restart lalu lanjutkan dari sel pertama.

In [ ]:
%pip -q install -U "transformers>=4.46,<5" "datasets>=3.0" "peft>=0.13" "accelerate>=1.1" "bitsandbytes>=0.44" "safetensors"
print('Instalasi selesai')

## 3. Muat model Qwen dalam 4-bit

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map='auto',
)
print('Model Qwen berhasil dimuat')

## 4. Uji respons model

In [ ]:
messages = [
    {'role': 'system', 'content': 'Kamu adalah AI Lara, asisten yang membantu merancang aplikasi Android.'},
    {'role': 'user', 'content': 'Jelaskan langkah awal membuat aplikasi catatan sederhana untuk Android.'},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=250, do_sample=True, temperature=0.7, top_p=0.9)
print(tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))

## 5. Siapkan dataset contoh untuk AI Lara
Sel ini hanya menyiapkan lima contoh instruksi-respons. Ini belum melatih model; kita akan menambahkan konfigurasi LoRA setelah dataset ini berhasil dibuat.

In [ ]:
from datasets import Dataset

data = [
    {
        'instruction': 'Buat aplikasi catatan sederhana untuk Android.',
        'response': 'Buat proyek Android dengan Kotlin. Tampilkan daftar catatan, kolom input, tombol simpan, dan penyimpanan lokal.'
    },
    {
        'instruction': 'Buat aplikasi daftar tugas Android.',
        'response': 'Buat aplikasi Android dengan Kotlin yang memiliki daftar tugas, tombol tambah, fitur tandai selesai, dan penyimpanan lokal.'
    },
    {
        'instruction': 'Bagaimana membuat APK Android?',
        'response': 'Siapkan proyek Android yang lengkap, gunakan Gradle, lalu jalankan tugas assembleDebug untuk membangun APK debug.'
    },
    {
        'instruction': 'Buat aplikasi kalkulator Android.',
        'response': 'Buat antarmuka kalkulator dengan tombol angka, operasi matematika, dan area hasil.'
    },
    {
        'instruction': 'Apa yang dibutuhkan AI Lara untuk menghasilkan APK?',
        'response': 'AI Lara perlu menghasilkan kode sumber proyek Android, menyiapkan konfigurasi build, menjalankan Gradle, lalu memeriksa berkas APK yang dihasilkan.'
    }
]

dataset = Dataset.from_list(data)
print('Jumlah contoh:', len(dataset))
print(dataset[0])

## Tahap berikutnya
Setelah model dan dataset contoh berhasil diuji, tahap berikutnya adalah menyiapkan format prompt dan konfigurasi pelatihan LoRA. Dataset lima contoh ini hanya untuk memastikan alur kerja berjalan; pelatihan yang bermanfaat membutuhkan lebih banyak contoh berkualitas. **Adapter LoRA bukan APK**; pembuatan APK Android adalah tahap proyek terpisah.